<a href="https://colab.research.google.com/github/wesleykoe/UNICC-Capstone/blob/main/UNICC_TRAINING_TEST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# CELL 1: Runtime & GPU Verification
# ============================================================
# Before anything else, verify Colab has assigned a GPU.
# If this shows CPU, go to Runtime > Change Runtime Type > GPU

import torch

print("=== Runtime Check ===")
print(f"PyTorch version:   {torch.__version__}")
print(f"CUDA available:    {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU device:        {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:        {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  WARNING: No GPU detected. Go to Runtime > Change Runtime Type > T4 GPU")

=== Runtime Check ===
PyTorch version:   2.10.0+cu128
CUDA available:    True
GPU device:        Tesla T4
GPU memory:        15.6 GB


In [2]:
# ============================================================
# CELL 2: Install Dependencies (No bitsandbytes)
# ============================================================

!pip install -q transformers
!pip install -q peft
!pip install -q datasets
!pip install -q accelerate
!pip install -q scipy

import torch
import transformers
import peft

print(f"✅ CUDA available:  {torch.cuda.is_available()}")
print(f"✅ transformers:    {transformers.__version__}")
print(f"✅ peft:            {peft.__version__}")
print(f"✅ torch:           {torch.__version__}")

✅ CUDA available:  True
✅ transformers:    4.38.2
✅ peft:            0.9.0
✅ torch:           2.10.0+cu128


In [34]:
# ============================================================
# CELL 3: Mount Google Drive
# ============================================================
# Mount your Google Drive so we can:
# 1. Read the training JSONL files
# 2. Save trained adapters back to Drive

from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE  = "/content/drive/MyDrive/Class/Capstone/Training_Data"
DATASET_DIR = DRIVE_BASE
ADAPTER_DIR = f"{DRIVE_BASE}/adapters"

# Create adapter output folders if they don't exist yet
os.makedirs(f"{ADAPTER_DIR}/governance_adapter",  exist_ok=True)
os.makedirs(f"{ADAPTER_DIR}/threat_adapter",      exist_ok=True)
os.makedirs(f"{ADAPTER_DIR}/behavioral_adapter",  exist_ok=True)

print(f"✅ Drive mounted")
print(f"   Dataset dir: {DATASET_DIR}")
print(f"   Adapter dir: {ADAPTER_DIR}")

# Verify dataset files exist before proceeding
for fname in ["governance_train.jsonl", "threat_train.jsonl", "behavioral_train.jsonl"]:
    path = f"{DATASET_DIR}/{fname}"
    exists = os.path.exists(path)
    print(f"   {'✅' if exists else '❌'} {fname}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted
   Dataset dir: /content/drive/MyDrive/Class/Capstone/Training_Data
   Adapter dir: /content/drive/MyDrive/Class/Capstone/Training_Data/adapters
   ✅ governance_train.jsonl
   ✅ threat_train.jsonl
   ✅ behavioral_train.jsonl


In [35]:
# ============================================================
# CELL 4: Imports & Global Configuration
# ============================================================
# Centralise all configuration in one place.
# To switch base models later (e.g. on DGX), change MODEL_NAME only.

import os
import json
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from datasets import load_dataset
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)

# ── Model ────────────────────────────────────────────────────
# Mistral-7B is our production base model.
# On DGX: remove quantization config and set dtype=torch.bfloat16
MODEL_NAME = "facebook/opt-1.3b"
MAX_LENGTH  = 1024

# ── LoRA Hyperparameters ─────────────────────────────────────
LORA_R            = 16
LORA_ALPHA        = 32
LORA_DROPOUT      = 0.05
LORA_TARGET_MODS  = ["q_proj", "v_proj"]

# ── Training Hyperparameters ─────────────────────────────────
BATCH_SIZE          = 2
GRAD_ACCUM_STEPS    = 4
NUM_EPOCHS          = 5
LEARNING_RATE       = 2e-4
WARMUP_STEPS        = 10

# ── Paths ────────────────────────────────────────────────────
DRIVE_BASE  = "/content/drive/MyDrive/Class/Capstone/Training_Data"
DATASET_DIR = DRIVE_BASE
ADAPTER_DIR = f"{DRIVE_BASE}/adapters"

print("✅ Configuration loaded")
print(f"   Base model:      {MODEL_NAME}")
print(f"   Max length:      {MAX_LENGTH}")
print(f"   LoRA rank:       {LORA_R}")
print(f"   Batch size:      {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM_STEPS})")
print(f"   Epochs:          {NUM_EPOCHS}")

✅ Configuration loaded
   Base model:      facebook/opt-1.3b
   Max length:      1024
   LoRA rank:       16
   Batch size:      2 (effective: 8)
   Epochs:          5


In [36]:
# ============================================================
# CELL 5: Model Loader (float16, No Quantization)
# ============================================================
# Instead of 4-bit QLoRA, we load in float16 directly.
# T4 has 16GB VRAM — Mistral-7B in float16 uses ~14GB.
# This is tight but workable, and avoids bitsandbytes entirely.

from google.colab import userdata
import huggingface_hub

hf_token = userdata.get('HF_TOKEN')
huggingface_hub.login(token=hf_token)

def load_base_model():
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        token=hf_token,
        use_fast=False
    )
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        device_map={"": 0},        # Force everything onto GPU 0, no offloading
        trust_remote_code=True,
        token=hf_token
    )

    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

    return model, tokenizer

print("✅ Model loader defined")

✅ Model loader defined


In [37]:
# ============================================================
# CELL 6: Dataset Formatting Function
# ============================================================
# Converts raw JSONL rows into tokenized training examples.
# Labels for prompt tokens are set to -100 so the model
# only learns to predict the OUTPUT portion, not the input.

def build_dataset(jsonl_path, tokenizer, expert_role):
    """
    Load and format a JSONL dataset for causal LM training.

    Args:
        jsonl_path:  Path to the training JSONL file
        tokenizer:   Loaded HuggingFace tokenizer
        expert_role: String label e.g. "Governance Expert"

    Returns:
        HuggingFace Dataset with input_ids, attention_mask, labels
    """

    raw_dataset = load_dataset(
        "json",
        data_files=jsonl_path,
        split="train"
    )

    def format_example(example):
        # Serialize input/output dicts to JSON strings
        input_str  = json.dumps(example["input"],  indent=2) \
                     if isinstance(example["input"],  dict) \
                     else str(example["input"])

        output_str = json.dumps(example["output"], indent=2) \
                     if isinstance(example["output"], dict) \
                     else str(example["output"])

        # Construct prompt — model learns to complete this with output
        prompt = (
            f"You are the {expert_role} in an AI Safety Evaluation Council.\n"
            f"Evaluate the following AI system and return structured JSON.\n\n"
            f"### INPUT:\n{input_str}\n\n"
            f"### OUTPUT:\n"
        )
        completion = output_str

        # Tokenize prompt alone to find where output begins
        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=True
        ).input_ids

        # Tokenize full sequence with truncation
        full_ids = tokenizer(
            prompt + completion,
            add_special_tokens=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).input_ids

        # Build labels — -100 masks prompt tokens from loss calculation
        labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

        # Pad to MAX_LENGTH for batching
        pad_len        = MAX_LENGTH - len(full_ids)
        input_ids      = full_ids + [tokenizer.pad_token_id] * pad_len
        attention_mask = [1] * len(full_ids) + [0] * pad_len
        labels         = labels + [-100] * pad_len

        return {
            "input_ids":       input_ids[:MAX_LENGTH],
            "attention_mask":  attention_mask[:MAX_LENGTH],
            "labels":          labels[:MAX_LENGTH]
        }

    dataset = raw_dataset.map(
        format_example,
        remove_columns=raw_dataset.column_names,
        batched=False
    )

    return dataset


print("✅ Dataset formatter defined")


✅ Dataset formatter defined


In [38]:
# ============================================================
# CELL 6B: Aggressively patch PEFT to disable bitsandbytes
# ============================================================

import unittest.mock as mock
import sys

# Create a fake bitsandbytes module that satisfies all PEFT checks
fake_bnb = mock.MagicMock()
fake_bnb.nn = mock.MagicMock()
fake_bnb.nn.Linear4bit = mock.MagicMock()
fake_bnb.nn.Linear8bitLt = mock.MagicMock()

# Inject fake module into sys.modules so any import of bitsandbytes
# returns our mock instead of the broken real one
sys.modules['bitsandbytes'] = fake_bnb
sys.modules['bitsandbytes.nn'] = fake_bnb.nn
sys.modules['bitsandbytes.optim'] = mock.MagicMock()

# Now patch PEFT's import checks to return False
import peft.import_utils as peft_utils
peft_utils.is_bnb_available = lambda: False
peft_utils.is_bnb_4bit_available = lambda: False

# Patch directly in peft.tuners.lora as well
import peft.tuners.lora.model as lora_model
lora_model.is_bnb_available = lambda: False
lora_model.is_bnb_4bit_available = lambda: False

print("✅ bitsandbytes fully mocked")
print(f"   is_bnb_available:      {peft_utils.is_bnb_available()}")
print(f"   is_bnb_4bit_available: {peft_utils.is_bnb_4bit_available()}")

✅ bitsandbytes fully mocked
   is_bnb_available:      False
   is_bnb_4bit_available: False


In [39]:
# ============================================================
# CELL 7: LoRA Configuration & Training Function
# ============================================================
# Reusable training function for all 3 experts.
# Each call: attaches fresh LoRA adapters, trains, saves,
# then frees GPU memory for the next expert.

def train_expert(expert_role, jsonl_path, output_dir):
    """
    Full training pipeline for one expert adapter.

    Args:
        expert_role: Display name e.g. "Governance Expert"
        jsonl_path:  Path to training JSONL file
        output_dir:  Path to save the trained LoRA adapter
    """

    print(f"\n{'='*60}")
    print(f"  TRAINING: {expert_role}")
    print(f"{'='*60}")

    # Step 1: Load fresh base model for this expert
    print("Loading base model...")
    model, tokenizer = load_base_model()

    # Step 2: Attach LoRA adapters
    # Only ~1-2% of parameters are trainable
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODS,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Step 3: Load and format dataset
    print(f"Loading dataset from {jsonl_path}...")
    dataset = build_dataset(jsonl_path, tokenizer, expert_role)
    print(f"Dataset ready: {len(dataset)} examples")

    # Step 4: Configure training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=NUM_EPOCHS,
        warmup_steps=WARMUP_STEPS,
        optim="adamw_torch",    # Memory-efficient optimizer for QLoRA
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        fp16=False,                   # Native fp16 on T4 = ~2x speedup
        bf16=False,
        save_strategy="steps",
        save_steps=25,
        save_total_limit=2,
        logging_steps=10,
        report_to="none",
        dataloader_pin_memory=True,
        dataloader_num_workers=2,
    )

    # Step 5: Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm=False
        )
    )

    # Step 6: Train
    print("Starting training...")
    trainer.train()

    # Step 7: Save adapter to Drive
    # Saves only LoRA weights (~50-100MB), not the full model
    print(f"Saving adapter to {output_dir}...")
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"✅ {expert_role} adapter saved")

    # Step 8: Free GPU memory before next expert
    del model
    del tokenizer
    torch.cuda.empty_cache()
    print(f"✅ GPU memory cleared")


print("✅ Training function defined")


✅ Training function defined


In [40]:
# ============================================================
# CELL 8: Train Governance Expert
# ============================================================
# Trains on S1-S30 (governance standalone) + SH1-SH30 (shared)
# Expected time on T4: ~5-10 minutes
# Watch loss — should decrease from ~2.0 toward ~0.5

train_expert(
    expert_role = "Governance Expert",
    jsonl_path  = f"{DATASET_DIR}/governance_train.jsonl",
    output_dir  = f"{ADAPTER_DIR}/governance_adapter"
)


  TRAINING: Governance Expert
Loading base model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


trainable params: 3,145,728 || all params: 1,318,903,808 || trainable%: 0.23851079820371554
Loading dataset from /content/drive/MyDrive/Class/Capstone/Training_Data/governance_train.jsonl...
Dataset ready: 60 examples
Starting training...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss
10,1.681400
20,1.233500


Step,Training Loss
10,1.681400
20,1.233500
30,0.914500


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Saving adapter to /content/drive/MyDrive/Class/Capstone/Training_Data/adapters/governance_adapter...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Governance Expert adapter saved
✅ GPU memory cleared


In [ ]:
# ============================================================
# CELL 9: Train Threat Expert
# ============================================================
# Trains on T1-T30 (threat standalone) + SH1-SH30 (shared)
# Expected time on T4: ~5-10 minutes
# GPU memory is cleared automatically after Cell 8 completes

train_expert(
    expert_role = "Threat Expert",
    jsonl_path  = f"{DATASET_DIR}/threat_train.jsonl",
    output_dir  = f"{ADAPTER_DIR}/threat_adapter"
)


  TRAINING: Threat Expert
Loading base model...
trainable params: 3,145,728 || all params: 1,318,903,808 || trainable%: 0.23851079820371554
Loading dataset from /content/drive/MyDrive/Class/Capstone/Training_Data/threat_train.jsonl...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Dataset ready: 60 examples
Starting training...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss
10,1.668700


In [ ]:
# ============================================================
# CELL 10: Train Behavioral Expert
# ============================================================
# Trains on B1-B30 (behavioral standalone) + SH1-SH30 (shared)
# Expected time on T4: ~5-10 minutes
# Final adapter — all 3 will be saved to Drive after this

train_expert(
    expert_role = "Behavioral Expert",
    jsonl_path  = f"{DATASET_DIR}/behavioral_train.jsonl",
    output_dir  = f"{ADAPTER_DIR}/behavioral_adapter"
)

In [ ]:
# ============================================================
# CELL 11: Verify All Adapters Saved to Drive
# ============================================================
# Confirms all 3 adapter folders exist and contain the
# expected files before closing the Colab session.
#
# Expected files per adapter:
#   adapter_config.json       — LoRA configuration
#   adapter_model.safetensors — Trained adapter weights
#   tokenizer.json            — Tokenizer files
#   tokenizer_config.json     — Tokenizer config

import os

print("=== Adapter Verification ===\n")

adapters = {
    "Governance": f"{ADAPTER_DIR}/governance_adapter",
    "Threat":     f"{ADAPTER_DIR}/threat_adapter",
    "Behavioral": f"{ADAPTER_DIR}/behavioral_adapter",
}

all_good = True
for name, path in adapters.items():
    if os.path.exists(path):
        files = os.listdir(path)
        has_weights   = any("adapter_model" in f for f in files)
        has_config    = "adapter_config.json" in files
        has_tokenizer = "tokenizer.json" in files
        status = "✅" if (has_weights and has_config) else "⚠️"
        if not (has_weights and has_config):
            all_good = False
        print(f"{status} {name} adapter")
        print(f"     Path:  {path}")
        print(f"     Files: {files}\n")
    else:
        print(f"❌ {name} adapter — folder not found at {path}")
        all_good = False

if all_good:
    print("✅ All 3 adapters verified — ready for evaluate_system.py")
else:
    print("⚠️  Some adapters missing — re-run the relevant training cell")